In [17]:
import os
from dotenv import load_dotenv
from pydantic import BaseModel

# load env vars
load_dotenv()

True

In [18]:
class MetricEvaluation(BaseModel):
    specific_metric_name: str
    evaluation_reasoning: str
    direct_quote_evidence: str | None
    failed_metric_check: bool

class Metric(BaseModel):
    evaluation_category: str
    specific_metric_name: str
    description: str

# Metrics generated by Gemini Pro with the prompt:
# Synthesize the insights from this paper into a prompt for a code review comment reviewer. Pull out specific aspects and heuristics this paper notes as being important for a good code review comment.

# civility
# Rahman, Md Shamimur, Zadia Codabux, and Chanchal K. Roy. "Do words have power? understanding and fostering civility in code review discussion." Proceedings of the ACM on Software Engineering 1.FSE (2024): 1632-1655.
civility_metrics = [
    Metric(
        evaluation_category="Civility",
        specific_metric_name="Explicit Attacks",
        description="Direct insults, name-calling, or unprofessional hostility."
    ),
    Metric(
        evaluation_category="Civility",
        specific_metric_name="Tonal Behaviors",
        description="Indicators of arrogance, impatience, entitlement, or frustration."
    ),
    Metric(
        evaluation_category="Civility",
        specific_metric_name="Nuanced Incivility",
        description="Subtle mockery, provocation, or disrespectful references to identity (gender, race, etc.)."
    ),
    Metric(
        evaluation_category="Civility",
        specific_metric_name="Competence Questioning",
        description='Derogatory remarks regarding basic skills or labeling work as subpar.'
    )
]

# Yang, Lanxin, et al. "Evacrc: Evaluating code review comments." Proceedings of the 31st ACM Joint European Software Engineering Conference and Symposium on the Foundations of Software Engineering. 2023.
quality_metrics = [
    Metric(
        evaluation_category="Quality",
        specific_metric_name="Emotion (Tone)",
        description="Assess the tone of the comment. Ensure it is respectful, constructive, and positive or neutral. Flag any negative, condescending, or aggressive language."
    ),
    Metric(
        evaluation_category="Quality",
        specific_metric_name="Question (Inquiry)",
        description="Evaluate if the comment effectively uses questions. Good comments often ask clarifying questions to understand the author's original intent or reasoning rather than making immediate assumptions."
    ),
    Metric(
        evaluation_category="Quality",
        specific_metric_name="Evaluation (Assessment)",
        description="Determine if the comment clearly identifies and explains a specific issue in the code (e.g., functional defects, security flaws, readability issues, or coding standard violations). Does it clearly state why something is an issue?"
    ),
    Metric(
        evaluation_category="Quality",
        specific_metric_name="Suggestion (Actionability)",
        description="Check whether the comment provides a concrete, actionable solution, alternative approach, or specific guidance to help the author fix the identified problem."
    )
]

# Rahman, Mohammad Masudur, Chanchal K. Roy, and Raula G. Kula. "Predicting usefulness of code review comments using textual features and developer experience." 2017 IEEE/ACM 14th International Conference on Mining Software Repositories (MSR). IEEE, 2017.
usefulness_metrics = [
    Metric(
        evaluation_category="Usefulness",
        specific_metric_name="Actionability",
        description="Does the comment warrant a clear action or suggest a specific code change within the immediate vicinity (1-10 lines) of the code? Evaluate if it moves beyond merely asking open-ended questions for clarification."
    ),
    Metric(
        evaluation_category="Usefulness",
        specific_metric_name="Relevant Code Elements",
        description="Does the comment explicitly reference salient code artifacts, such as specific program entity names, methods, or variables present in the patch?"
    ),
    Metric(
        evaluation_category="Usefulness",
        specific_metric_name="Conceptual Similarity",
        description="Does the feedback share vocabulary and conceptually align with the specific lines of code being changed?"
    ),
    Metric(
        evaluation_category="Usefulness",
        specific_metric_name="Conciseness",
        description="Is the comment direct and focused? Evaluate whether it minimizes unnecessary filler words to deliver the feedback efficiently."
    ),
    Metric(
        evaluation_category="Usefulness",
        specific_metric_name="Directness",
        description="Does the comment rely too heavily on questions rather than providing clear, actionable statements? (Comments that simply ask for clarification without suggesting a path forward are statistically less useful)."
    )
]

# Chen, Junkai, et al. "Understanding practitioners’ expectations on clear code review comments." Proceedings of the ACM on Software Engineering 2.ISSTA (2025): 1257-1279.
clarity_metrics = [
    Metric(
        evaluation_category="Clarity",
        specific_metric_name="Relevance",
        description="Does the code review comment directly address the specific code change being reviewed? Ensure the feedback is highly contextualized to the lines of code, architecture, or logic modified in the pull request, avoiding off-topic remarks."
    ),
    Metric(
        evaluation_category="Clarity",
        specific_metric_name="Informativeness",
        description="Does the comment provide sufficient information for the author to understand the issue and know how to fix it? If the comment is rejecting a change or pointing out a flaw, does it at least include a suggestion for an alternative approach? Does the comment include necessary pointers, references to documentation, or links to existing discussions to support the feedback?"
    ),
    Metric(
        evaluation_category="Clarity",
        specific_metric_name="Expression",
        description="Is the comment written in a way that is easily digestible and positively received? Check that the language is clear, readable, easy to understand, and maintains a friendly, constructive tone. Avoid overly critical or ambiguous phrasing."
    )
]

reflection_stop_word = "<good>"

def reflection_prompt_factory(generation_prompt):
    reflection_prompt = f"""
    Check each that the message has every aspect in this list covered.

    {generation_prompt}

    Double check that the recommendation evaluated across each of the numbered aspects. Output {reflection_stop_word} only if all elements are covered. If it needs editing. Respond with the items that were missed.
    """

    return reflection_prompt

summary_prompt = f"""
Looking over the conversation history, output the generation unit's suggested revision for improving the quality of the code review comment from the user.
"""



In [23]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient


model_client = OpenAIChatCompletionClient(
    model="gpt-4o",
    api_key= os.getenv("OPENAI_API_KEY")
)


async def eval_metric(metric: Metric, crc_text: str):
    evaluator_agent = AssistantAgent(
        name="metric_evaluator",
        model_client=model_client,
        system_message="You are an expert code review evaluator. Assess the provided comment against the specific metric and output strictly matching the requested JSON schema.",
        output_content_type =MetricEvaluation
    )

    task_prompt = f"""
        Evaluate this code review comment:
        "{crc_text}"

        Category: {metric.evaluation_category}
        Metric: {metric.specific_metric_name}
        Definition: {metric.description}
        """

    result = await evaluator_agent.run(task=task_prompt)

    response_text = result.messages[-1].content
    evaluation = MetricEvaluation.model_validate(response_text)
    return evaluation


async def synthesize_deficiencies(evaluation_category, failed_evals: list[MetricEvaluation], crc_text: str) -> str:
    if not failed_evals:
        return "No deficiencies found."

    metrics_context = "\n".join(
        [f"- [{evaluation_category} | {eval.specific_metric_name}]: {eval.evaluation_reasoning} (Evidence: '{eval.direct_quote_evidence}')"
         for eval in failed_evals]
    )

    synthesis_prompt = f"""
    You are a strict code review auditor. Your sole objective is to synthesize a list of failed evaluation metrics into a succinct, focused summary of a code review comment's deficiencies.

    # Constraints
    - DIAGNOSE ONLY: You must only describe what is wrong with the comment.
    - NO SOLUTIONS: Do absolutely NOT suggest solutions, improvements, or alternative phrasing.
    - NO REWRITES: Do NOT attempt to rewrite the comment into a better version.
    - Be concise, analytical, and objective. Group related failures together.

    # Context
    Original Code Review Comment:
    "{crc_text}"

    Failed Metrics & Evidence:
    {metrics_context}
    """

    synthesis_agent = AssistantAgent(name="deficiency_summarizer", model_client=model_client)
    result = await synthesis_agent.run(task=synthesis_prompt)

    return result.messages[-1].content

async def generate_recommendations(deficiencies_summary: list[str], crc_text: str) -> str:
    if deficiencies_summary == "No deficiencies found.":
        return "The original comment meets all quality and civility standards."

    summary_pretty = [f"{n}. {deficiency}" for n, deficiency in enumerate(deficiencies_summary)]

    recommendation_prompt = f"""
    You are a Senior Staff Engineer mentoring a team member on effective code review communication. Your goal is to help them improve a poorly written code review comment based on a provided diagnostic summary.

    # Constraints
    - FOCUS ON ACTION: Provide 2-3 specific, actionable recommendations on how to fix the identified issues.
    - BE CONSTRUCTIVE: Maintain an encouraging and professional tone.
    - PROVIDE AN EXAMPLE: Conclude with a single, highly effective rewritten version of the original comment that resolves all deficiencies.
    - FORMAT: Use clear bullet points for the recommendations, followed by a "Suggested Revision:" block.

    # Context
    Original Code Review Comment:
    "{crc_text}"

    Identified Deficiencies:
    "{'\n'.join(summary_pretty)}"
    """

    mentor_agent = AssistantAgent(name="recommendation_mentor", model_client=model_client)
    result = await mentor_agent.run(task=recommendation_prompt)

    return result.messages[-1].content


dimensions = [ civility_metrics, quality_metrics , usefulness_metrics, clarity_metrics ]

# Run the agent and stream the messages to the console.
async def main() -> None:
    crc_example = "Why are you continuing to use this depreciated method when we have discussed the alternatives. You need to come ask me questions if you are confused again, this is a waste of my time and company money."

    failed_category_summaries = []
    for dimension in dimensions:
        failed_evaluations = []
        print(f"\n\n{dimension[0].evaluation_category}--------------------------------------")
        for metric in dimension:
            evaluation = await eval_metric(metric,crc_example)

            if evaluation.failed_metric_check:
                print(f"CRC failed: {evaluation.specific_metric_name}: {evaluation.evaluation_reasoning}\n")
                failed_evaluations.append(evaluation)


        if not failed_evaluations:
            print(f"No deficiencies found in dimension {dimension[0].dimension_name}")
            continue

        print(f"{len(failed_evaluations)} issues detected in dimension {dimension[0].evaluation_category}")
        print(f"\n{dimension[0].evaluation_category} Summary:--------------------------------------")


        synthesis = await synthesize_deficiencies(dimension[0].evaluation_category,failed_evaluations, crc_example)
        print(synthesis)

        failed_category_summaries.append(synthesis)

    print("\n\nGenerating suggestion-------------------------------------")
    suggestion = await generate_recommendations(failed_category_summaries, crc_example)
    print(suggestion)

    # Close the connection to the model client.
    await model_client.close()



await main()




Civility--------------------------------------
CRC failed: Tonal Behaviors: The comment demonstrates impatience and frustration through its direct and accusatory language. The choice of words like 'Why are you continuing', 'we have discussed', and 'this is a waste of my time and company money' indicate a tone of arrogance and entitlement. The reviewer implies that the author should already know better or should have resolved the issue without requiring further assistance. The use of phrases that suggest the work is a waste of resources reflects frustration, further straying from the civility metric of maintaining a constructive and supportive tone.

CRC failed: Nuanced Incivility: The comment does display elements of nuanced incivility. While it doesn't contain direct mockery or clear references to identity, it does imply disrespect by accusing the recipient of wasting time and money. The directive to "come ask me questions" in a context indicating previous discussions could be seen 